# Best Assistors Leaderboard Analysis

This notebook processes player data to generate a comprehensive best assistors leaderboard that emphasizes both individual performance and team success factors.

## Section 1: Import Required Libraries

Import pandas and numpy for data processing and manipulation.

In [ ]:
import pandas as pd
import numpy as np
import os

# Set the working directory to player_stats folder
os.chdir(r'c:\Users\vardh\Desktop\MY.PROJECTS\prediction.oc\Prediction.oc\player_stats')

print("=== PROCESSING BEST ASSISTOR LEADERBOARD (DOMINATING TEAM POINTS) ===")
print("Libraries imported successfully!")

## Section 2: Load All Data Files

Load all CSV files including player stats, shooting data, players data, and tournament information.

In [ ]:
# Load all files to ensure complete cross-referencing
df_stats = pd.read_csv('cleaned_player_stats.csv')
df_shooting = pd.read_csv('cleaned_player_shooting.csv')
df_data = pd.read_csv('cleaned_players_data-2024_2025.csv')
df_goals = pd.read_csv('goals_cleaned.csv')
# Load structural files to maintain system pipeline consistency
df_players = pd.read_csv('cleaned_players.csv')
df_squads = pd.read_csv('cleaned_squads.csv')
df_tournaments = pd.read_csv('cleaned_tournaments.csv')
df_standings = pd.read_csv('cleaned_predicted_group_standings.csv')
df_probs = pd.read_csv('cleaned_team_position_probabilities.csv')

print("All data files loaded successfully!")
print(f"Player stats shape: {df_stats.shape}")
print(f"Shooting data shape: {df_shooting.shape}")
print(f"Standings data shape: {df_standings.shape}")

## Section 3: Merge and Clean Data

Merge player stats and shooting data tables, then clean null values for assist-related columns.

In [ ]:
# Merge tournament stats and shooting tables
df_merged = pd.merge(df_stats, df_shooting, on=['player', 'team'], suffixes=('_stats', '_shooting'), how='outer')

print(f"Merged data shape: {df_merged.shape}")

# Clean null values
df_merged['assists'] = df_merged['assists'].fillna(0)
df_merged['xg_assist'] = df_merged['xg_assist'].fillna(0)
df_merged['xg_assist_per90'] = df_merged['xg_assist_per90'].fillna(0)
df_merged['minutes'] = df_merged['minutes'].fillna(0)

print("\nAssist-related columns cleaned:")
print(f"Total assists: {df_merged['assists'].sum()}")
print(f"Average xG Assist per 90: {df_merged['xg_assist_per90'].mean():.4f}")
print(f"Data cleaning completed!")

## Section 4: Calculate Assistor Scores with Team Context

Create a composite scoring formula that emphasizes:
- Assists (40% weight): Direct assist count
- Expected Assisted Goals (20% weight): Quality of chances created
- Minutes Played (0.05% weight): Time on the field
- Team Points (15% weight): Team success bonus rewarding players from dominant teams

In [ ]:
# Extract team points from predicted standings
pts_map = df_standings.groupby('team')['points'].max().to_dict()

print(f"Teams in standings: {len(pts_map)}")
print(f"Points range: {min(pts_map.values())} to {max(pts_map.values())}")

# Scoring formula emphasizing dominating team success weight (0.15 * Team Points)
# This heavily rewards teams with flawless standings records
df_merged['assistor_score'] = (
    (df_merged['assists'] * 0.40) + 
    (df_merged['xg_assist'] * 0.20) + 
    (df_merged['minutes'] * 0.0005) + 
    (df_merged['team'].map(lambda t: pts_map.get(t, 0)) * 0.15)
)

print("\nAssistor Score Calculation Complete!")
print(f"Score range: {df_merged['assistor_score'].min():.4f} to {df_merged['assistor_score'].max():.4f}")
print(f"Average score: {df_merged['assistor_score'].mean():.4f}")

## Section 5: Generate Best Assistors Leaderboard

Create a ranked leaderboard of top assistors sorted by their composite assistor score.

In [ ]:
# Determine position column name
position_col = 'position_stats' if 'position_stats' in df_merged.columns else 'position'

# Format output columns
output_cols = ['player', 'team', position_col, 'assists', 'xg_assist', 'xg_assist_per90', 'minutes', 'assistor_score']
available_cols = [col for col in output_cols if col in df_merged.columns]

print(f"Available columns: {available_cols}")

leaderboard = df_merged[available_cols].copy()
leaderboard = leaderboard.rename(columns={position_col: 'position', 'minutes': 'minutes_played'})

# Rank and sort players
leaderboard = leaderboard.sort_values(by='assistor_score', ascending=False).reset_index(drop=True)
leaderboard['rank'] = leaderboard.index + 1

final_columns = ['rank', 'player', 'team', 'position', 'assists', 'xg_assist', 'xg_assist_per90', 'minutes_played', 'assistor_score']
leaderboard = leaderboard[[col for col in final_columns if col in leaderboard.columns]]

print(f"\nBest Assistors Leaderboard Generated!")
print(f"Total players ranked: {len(leaderboard)}")
print(f"Leaderboard shape: {leaderboard.shape}")

## Section 6: Save Results and Display Top Assistors

Export the final leaderboard to CSV file and display the top 20 players.

In [ ]:
# Export to CSV
output_file = 'best_assistors_leaderboard.csv'
leaderboard.to_csv(output_file, index=False)
print(f"✓ Leaderboard saved to: {output_file}")

# Display top 20 assistors
print("\n" + "="*120)
print("TOP 20 BEST ASSISTORS (DOMINATING TEAM POINTS WEIGHTED)")
print("="*120)
display_cols = ['rank', 'player', 'team', 'position', 'assists', 'xg_assist', 'xg_assist_per90', 'assistor_score']
top_20 = leaderboard[display_cols].head(20)
print(top_20.to_string(index=False))

# Additional statistics
print("\n" + "="*120)
print("LEADERBOARD STATISTICS")
print("="*120)
print(f"Total players ranked: {len(leaderboard)}")
print(f"Top assistor score: {leaderboard['assistor_score'].max():.4f}")
print(f"Average assistor score: {leaderboard['assistor_score'].mean():.4f}")
print(f"Median assistor score: {leaderboard['assistor_score'].median():.4f}")
print(f"Total assists in dataset: {leaderboard['assists'].sum():.0f}")
print(f"Average assists per player: {leaderboard['assists'].mean():.2f}")

print("\nProcessing Complete! ✓")